# CAMELS: Time Series for the Website
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 27-08-2026<br>

**Introduction:**<br>
This script combines the daily discharge records with the daily basin meteorology and exports a time series per gauging station to be plotted in the website.

In [1]:
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

import logging
logger = logging.getLogger(__name__)

from ocab.config import Config
import ocab.variables as VARS
import ocab.meteorology as METEO
from ocab.plots.stations import plot_station_timeseries, create_station_html
from ocab.plots.utils import compute_climatology


## Configuration


In [2]:
cfg = Config('config_CAMELS_v200.yml')

# LSTM model
# model = 'bs256_dlr_se3ly_de2ly_2206_104834' # 495 catchments
model = '651_hs64_2708_153243'

# input paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'
path_results = cfg.path_dataset / 'results' / model if model is not None else None

# output paths
path_web = Path('../../docs')
path_layers = path_web / 'layers'
path_ts = path_web / 'timeseries' / 'stations'
path_plots = path_ts / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)
print(f'GeoJSON layers will be saved in:\t{path_layers}')
print(f'Parquet time series will be saved in:\t{path_ts}')
print(f'HTML plots will be saved in:\t\t{path_plots}')

# point layer
filename = 'stations.geojson'

GeoJSON layers will be saved in:	../../docs/layers
Parquet time series will be saved in:	../../docs/timeseries/stations
HTML plots will be saved in:		../../docs/timeseries/stations/plots



## Create time series


In [3]:
# load points
points = gpd.read_file(cfg.path_gis / filename).set_index('id')
print(f'{len(points)} basin outlets')

# add performance
performance_file = path_results / 'performance.geojson'
if performance_file.is_file():
    # read performance values
    performance = gpd.read_file(performance_file)
    performance.rename(columns={'gauge_id': 'id'}, inplace=True)
    performance.set_index('id', inplace=True)
    performance = performance[~performance.index.duplicated(keep='first')]
    performance = performance.loc[performance.index.intersection(points.index)]
    # concatenate to points
    points = pd.concat([points, performance.drop(columns='geometry')], axis=1)
else:
    logger.warning(f'The file {performance_file} does not exist')

1116 basin outlets


In [4]:

# process timeseries for each station
for ID in tqdm(points.index, desc='points'):
    
    # observed discharge timeseries
    try:
        observation = pd.read_parquet(path_in / 'discharge' / f'{ID}.parquet')
        observation.columns = ['discharge_cms']
        # compute specific discharge (mm/day)
        observation['discharge_mm'] = observation['discharge_cms'] / points.loc[ID, 'catch_skm'] * 86400 / 1000
        # round
        observation = observation[observation.columns.intersection(VARS.DECIMALS)].round(VARS.DECIMALS)
    except Exception as e:
        logger.error(f'Loading discharge timeseries for station {ID:04d}: {e}')
        continue

    # simulated discharge timeseries
    try:
        simulation_file = path_results / f'{ID}.parquet'
        if simulation_file.is_file():
            simulation = pd.read_parquet(simulation_file)
            cols = [col for col in simulation if col.endswith('sim')]
            simulation = simulation[cols]
            # round
            observation = observation[observation.columns.intersection(VARS.DECIMALS)].round(VARS.DECIMALS)
        else:
            simulation = pd.DataFrame()
    except Exception as e:
        logger.error(f'Loading simulated discharge timeseries for station {ID:04d}: {e}')

    # meteo timeseries
    try:
        meteo = {}
        for dataset, label in METEO.DATASETS.items():
            df = pd.read_parquet(path_in / 'meteo' / dataset / f'{ID}.parquet').loc[ID]
            # correct index
            df.index.name = 'date'
            if dataset in METEO.OFFSET_HOURS:
                df.index = df.index.date + pd.Timedelta(hours=METEO.OFFSET_HOURS[dataset])
            df.index = pd.to_datetime(df.index)
            # rename and round variables
            df.rename(columns=VARS.RENAME, inplace=True, errors='ignore')
            df = df[df.columns.intersection(VARS.DECIMALS)].round(VARS.DECIMALS)
            # # ensure average temperature exists
            # if 'temp_degC' not in df.columns:
            #     df['temp_degC'] = df[['temp_max_degC', 'temp_min_degC']].mean(axis=1)
            #     print(dataset)
            # add dataset name to columns
            df.columns = [f'{col}_{label}' for col in df.columns]
            # meteo.append(df)
            meteo[label] = df
        # meteo = pd.concat(meteo, axis=1)
    except Exception as e:
        logger.error(f'Loading meteo timeseries for station {ID}: {e}')
        # continue
    
    # merge timeseries
    start = max(
        cfg.start, 
        min([df.first_valid_index() for df in meteo.values()]),
    )
    start_ts = max(
        start, 
        observation.first_valid_index() - pd.Timedelta(days=365)
    )
    start_html = max(
        start, 
        observation.first_valid_index()
    )
    end = min(
        cfg.end, 
        max([df.last_valid_index() for df in meteo.values()]),
        observation.last_valid_index()
    )
    ts = [observation.loc[start_ts:end]]
    if simulation_file.is_file():
        ts.append(simulation)
    ts += [df.loc[start_ts:end] for df in meteo.values()]
    ts = pd.concat(ts, axis=1, sort=True)

    # # export timeseries
    ts.to_parquet(path_ts / f'{ID:04d}.parquet')

    # compute climatological values
    climatology = compute_climatology(ts)
    # mask = ~climatology.index.str.startswith('temp')
    # climatology[mask] = climatology[mask].round(0).astype(int)
    points.loc[ID, climatology.index] = climatology

    try:
        # extract attributes
        attrs = points.loc[ID]

        # create time series plot
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title() if pd.notna(attrs['name']) else '', 
            attrs['river'].title() if pd.notna(attrs['river']) else '', 
            attrs['basin'].title()
        )
        st, en = ts['discharge_mm'].first_valid_index(), ts['discharge_mm'].last_valid_index()
        fig = plot_station_timeseries(
            ts.loc[st:en],
            attrs,
            title=title,
            regime=attrs['regime'],
            save=True
        )
    except Exception as e:
        print(f"The plot for time series {ID} couldn't be created: {e}")
        break

    try:
        # save plot as HTML
        create_station_html(
            fig,
            path=path_plots / f'{ID}.html', 
            start=start_html.strftime('%Y-%m-%d'), 
            end=ts.index.max().strftime('%Y-%m-%d')
        )
    except Exception as e:
        print(f"The HTML for time series {ID} couldn't be created: {e}")

# export updated point layer
points.to_file(path_layers / filename)

points:   0%|          | 0/1116 [00:00<?, ?it/s]